# NLP Lab Assignment 4: Text and Script Normalization for Indic Languages

## Q1. NLTK Resources for Basic Analysis of Indic Language Text

In [1]:
import nltk
import unicodedata
import re
import pandas as pd
import numpy as np
from collections import Counter

# Download NLTK Indian corpus
nltk.download('indian', quiet=True)
from nltk.corpus import indian


In [2]:
# (a) & (b) Load Indian language text and extract at least 500 sentences
hindi_sentences = indian.sents('hindi.pos')[:500]

print(f"Extracted {len(hindi_sentences)} sentences from NLTK Indian Corpus (Hindi).")
print("Sample sentence:", " ".join(hindi_sentences[0]))


Extracted 500 sentences from NLTK Indian Corpus (Hindi).
Sample sentence: पूर्ण प्रतिबंध हटाओ : इराक


In [3]:
# (c) Perform tokenization and basic preprocessing
# Strip punctuation marks like |, ।, :, commas, etc. while preserving Indic matras
punct_chars = "।,;:!?()[]{}|./«»\'\""

all_tokens = []
for sent in hindi_sentences:
    for word in sent:
        clean_word = word.strip(punct_chars).strip()
        if clean_word:
            all_tokens.append(clean_word)

print("First 15 preprocessed tokens:")
print(all_tokens[:15])


First 15 preprocessed tokens:
['पूर्ण', 'प्रतिबंध', 'हटाओ', 'इराक', 'संयुक्त', 'राष्ट्र', 'इराक', 'के', 'विदेश', 'मंत्री', 'ने', 'अमरीका', 'के', 'उस', 'प्रस्ताव']


In [4]:
# (d) Calculate total tokens and vocabulary size
total_tokens = len(all_tokens)
vocabulary = sorted(list(set(all_tokens)))
vocab_size = len(vocabulary)
ttr = vocab_size / total_tokens

print(f"Total Tokens: {total_tokens}")
print(f"Vocabulary Size: {vocab_size}")
print(f"Type-Token Ratio (TTR): {ttr:.4f}")


Total Tokens: 8105
Vocabulary Size: 2084
Type-Token Ratio (TTR): 0.2571


In [5]:
# (e) Display the ten most frequent tokens after preprocessing
token_counts = Counter(all_tokens)
top_10 = token_counts.most_common(10)

top_10_df = pd.DataFrame(top_10, columns=["Token", "Frequency"])
print("Ten Most Frequent Tokens:")
print(top_10_df)


Ten Most Frequent Tokens:
  Token  Frequency
0    के        359
1   में        256
2    की        209
3    ने        196
4    है        189
5    को        150
6    से        131
7    पर        120
8    का        119
9    और        108


### Observations on Q1: Challenges of Tokenization and Preprocessing for Indic Languages

1. **Purna Viram Attachment:** The sentence-ending mark (`।`) is frequently attached directly to the last word without a space, causing splitters to misread it as part of the word.
2. **Agglutination and Sandhi:** Indic languages merge root words and suffixes (or combine two words into one, like विद्या + आलय = विद्यालय). Simple whitespace tokenization fails to segment them.
3. **Rich Morphology & Inflections:** Words change forms heavily based on gender, number, and case (लड़का, लड़के, लड़कों). Without a lemmatizer, each form creates a separate vocabulary entry, leading to high vocabulary sparsity.
4. **Lack of Capitalization:** Indic scripts lack uppercase/lowercase letters, which makes identifying named entities (proper nouns) harder without contextual features.

## Q2. Implement Text Normalization for Indic Language Text

In [6]:
# (a) Create at least 20 noisy Indic language text samples
noisy_samples = [
    "नमस्ते       दुनिया   !!!!!",
    "बहुतtttttt    अच्छाaaaa  काम   किया...",
    "कल  college  में   exam   है  ???",
    "यह   किताब   बहुत    महंगी   है।।।।",
    "आज   का   weather   बहुत   cool   है  :)",
    "train   letttt   हो   गयी   थी   !!",
    "कृपया    यहाँ    बैठिए   .....",
    "AI   lab   का   काम   बाकी   है   १२३",
    "भारत   का   स्कोर   ३५०   रन   था  |",
    "यह   मोबाइल   Phone   नया   है   100%",
    "शाबाश     दोस्त    वाह   वाह   वाह   !!!!!",
    "गाड़ी     धीमे     चलाओ     ...   .",
    "Data   Science   की   class   ८   बजे   है  .",
    "वह   office   पहुँच   गया   क्या   ??",
    "पानी     पीना    स्वास्थ्य   के   लिए   अच्छा   है  |||",
    "मेरे   पास   ५   tickets   हैं  ...",
    "party   में   खाना   bahut   delicious   था   !!!",
    "laptop   का   screen   खराब   हो   गया   १००",
    "चलो     घर     चलते     हैं     ।",
    "movie   का   review   सुपर   hit   था   :D"
]

print(f"Total noisy samples collected: {len(noisy_samples)}")


Total noisy samples collected: 20


In [7]:
# (b), (c), (d) Normalization function
indic_to_arabic_digits = str.maketrans('०१२३४५६७८९', '0123456789')

def normalize_indic_text(text):
    # (b) Unicode normalization (NFC)
    text = unicodedata.normalize('NFC', text)
    
    # (d) Case normalization: lowercase English / Roman-script words
    text = text.lower()
    
    # (b) Remove repeated characters (3 or more occurrences -> reduced to 1)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # (c) Normalize punctuation (convert '.' and '|' to standard Indic '।')
    text = re.sub(r'[\|.]+', '।', text)
    text = re.sub(r'[?!]+', '?', text)
    text = re.sub(r'[:;)(\[\]{}]', '', text)
    
    # (c) Normalize digits (convert Hindi numerals to standard digits)
    text = text.translate(indic_to_arabic_digits)
    
    # (b) Remove unwanted excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

normalized_samples = [normalize_indic_text(sample) for sample in noisy_samples]


In [8]:
# (e) Display original and normalized text side by side
df_q2 = pd.DataFrame({
    'Original (Noisy)': noisy_samples,
    'Normalized': normalized_samples
})
pd.set_option('display.max_colwidth', None)
print("Original vs Normalized Samples:")
print(df_q2)


Original vs Normalized Samples:
                                           Original (Noisy)  \
0                               नमस्ते       दुनिया   !!!!!   
1                    बहुतtttttt    अच्छाaaaa  काम   किया...   
2                         कल  college  में   exam   है  ???   
3                       यह   किताब   बहुत    महंगी   है।।।।   
4                  आज   का   weather   बहुत   cool   है  :)   
5                       train   letttt   हो   गयी   थी   !!   
6                            कृपया    यहाँ    बैठिए   .....   
7                     AI   lab   का   काम   बाकी   है   १२३   
8                      भारत   का   स्कोर   ३५०   रन   था  |   
9                     यह   मोबाइल   Phone   नया   है   100%   
10               शाबाश     दोस्त    वाह   वाह   वाह   !!!!!   
11                      गाड़ी     धीमे     चलाओ     ...   .   
12            Data   Science   की   class   ८   बजे   है  .   
13                    वह   office   पहुँच   गया   क्या   ??   
14  पानी     पीना    स्

### Observations on Q2: Importance of Text Normalization

1. **Noise Reduction:** Normalizing repeated characters and multiple spaces prevents spelling variations from inflating the vocabulary.
2. **Canonical Equivalence:** Unicode normalization (NFC) ensures composed and decomposed characters (like nukta combinations) share the same binary representation.
3. **Cross-Script Consistency:** Converting script-specific numerals (१, २, ३) to Arabic numerals (1, 2, 3) and standardizing punctuation creates consistent feature vectors for downstream NLP tasks.

## Q3. Script Normalization and Transliteration Analysis for Indic Languages

In [9]:
# (a) Select text samples written in at least two Indic scripts (Devanagari and Bengali)
devanagari_samples = [
    "भारत एक महान देश है",
    "विद्यार्थी पुस्तक पढ़ रहे हैं",
    "प्रकृति बहुत सुंदर है",
    "ज्ञान ही सबसे बड़ी शक्ति है"
]

bengali_samples = [
    "ভারত একটি মহান দেশ",
    "ছাত্ররা বই পড়ছে",
    "প্রকৃতি খুব সুন্দর",
    "জ্ঞানই সবচেয়ে বড় শক্তি"
]

print("Devanagari text samples:", devanagari_samples)
print("Bengali text samples:", bengali_samples)


Devanagari text samples: ['भारत एक महान देश है', 'विद्यार्थी पुस्तक पढ़ रहे हैं', 'प्रकृति बहुत सुंदर है', 'ज्ञान ही सबसे बड़ी शक्ति है']
Bengali text samples: ['ভারত একটি মহান দেশ', 'ছাত্ররা বই পড়ছে', 'প্রকৃতি খুব সুন্দর', 'জ্ঞানই সবচেয়ে বড় শক্তি']


In [10]:
# (b) & (c) Script normalization: Apply Unicode normalization
dev_normalized = [unicodedata.normalize('NFC', s) for s in devanagari_samples]
ben_normalized = [unicodedata.normalize('NFC', s) for s in bengali_samples]

# (d) Transliteration to a common Latin phonetic representation
# Mapping Devanagari and Bengali characters sharing Brahmi phonetic origins to Latin
phonetic_map = {
    # Devanagari consonants
    'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
    'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'ny',
    'ट': 't', 'ठ': 'th', 'ड': 'd', 'ढ': 'dh', 'ण': 'n',
    'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
    'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
    'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'v',
    'श': 'sh', 'ष': 'sh', 'स': 's', 'ह': 'h',
    'ड़': 'r', 'ढ़': 'rh', 'ज़': 'z', 'फ़': 'f',
    # Devanagari vowels & matras
    'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ee', 'उ': 'u', 'ऊ': 'oo', 'ऋ': 'ri', 'ए': 'e', 'ऐ': 'ai', 'ओ': 'o', 'औ': 'au',
    'ा': 'aa', 'ि': 'i', 'ी': 'ee', 'ु': 'u', 'ू': 'oo', 'ृ': 'ri', 'े': 'e', 'ै': 'ai', 'ो': 'o', 'ौ': 'au',
    '्': '', 'ं': 'n', 'ँ': 'n', 'ः': 'h',
    
    # Bengali consonants
    'ক': 'k', 'খ': 'kh', 'গ': 'g', 'ঘ': 'gh', 'ঙ': 'ng',
    'চ': 'ch', 'ছ': 'chh', 'জ': 'j', 'ঝ': 'jh', 'ঞ': 'ny',
    'ট': 't', 'ঠ': 'th', 'ড': 'd', 'ঢ': 'dh', 'ণ': 'n',
    'ত': 't', 'থ': 'th', 'দ': 'd', 'ধ': 'dh', 'ন': 'n',
    'প': 'p', 'ফ': 'ph', 'ব': 'b', 'ভ': 'bh', 'ম': 'm',
    'য': 'y', 'র': 'r', 'ল': 'l', 'শ': 'sh', 'ষ': 'sh', 'স': 's', 'হ': 'h',
    'ড়': 'r', 'ঢ়': 'rh', 'য়': 'y',
    # Bengali vowels & matras
    'অ': 'o', 'আ': 'aa', 'ই': 'i', 'ঈ': 'ee', 'উ': 'u', 'ঊ': 'oo', 'ঋ': 'ri', 'এ': 'e', 'ঐ': 'oi', 'ও': 'o', 'ঔ': 'ou',
    'া': 'aa', 'ি': 'i', 'ী': 'ee', 'ু': 'u', 'ূ': 'oo', 'ৃ': 'ri', 'ে': 'e', 'ৈ': 'oi', 'ো': 'o', 'ৌ': 'ou',
    '্': '', 'ং': 'ng', 'ঁ': 'n', 'ঃ': 'h'
}

def transliterate_to_latin(text):
    result = []
    for char in text:
        result.append(phonetic_map.get(char, char))
    return "".join(result)

dev_transliterated = [transliterate_to_latin(s) for s in dev_normalized]
ben_transliterated = [transliterate_to_latin(s) for s in ben_normalized]


In [11]:
# (e) Compare original, normalized, and converted text
comparison_q3 = pd.DataFrame({
    'Script': ['Devanagari']*len(devanagari_samples) + ['Bengali']*len(bengali_samples),
    'Original Text': devanagari_samples + bengali_samples,
    'Normalized Text': dev_normalized + ben_normalized,
    'Transliterated (Latin)': dev_transliterated + ben_transliterated
})

print("Script Normalization and Transliteration Comparison:")
print(comparison_q3)


Script Normalization and Transliteration Comparison:
       Script                  Original Text                Normalized Text  \
0  Devanagari            भारत एक महान देश है            भारत एक महान देश है   
1  Devanagari  विद्यार्थी पुस्तक पढ़ रहे हैं  विद्यार्थी पुस्तक पढ़ रहे हैं   
2  Devanagari          प्रकृति बहुत सुंदर है          प्रकृति बहुत सुंदर है   
3  Devanagari    ज्ञान ही सबसे बड़ी शक्ति है    ज्ञान ही सबसे बड़ी शक्ति है   
4     Bengali             ভারত একটি মহান দেশ             ভারত একটি মহান দেশ   
5     Bengali               ছাত্ররা বই পড়ছে               ছাত্ররা বই পড়ছে   
6     Bengali             প্রকৃতি খুব সুন্দর             প্রকৃতি খুব সুন্দর   
7     Bengali       জ্ঞানই সবচেয়ে বড় শক্তি       জ্ঞানই সবচেয়ে বড় শক্তি   

            Transliterated (Latin)  
0         bhaart ek mhaan desh hai  
1  vidyaarthee pustk pdh़ rhe hain  
2           prkriti bhut sundr hai  
3  jnyaan hee sbse bd़ee shkti hai  
4           bhaart ekti mhaan desh  
5            

### Observations on Q3: Role of Script Normalization in Multilingual and Cross-Lingual NLP

1. **Common Phonetic Foundation:** Most Indic scripts belong to the Brahmi family and share the same phonetic structure.
2. **Unified Representation:** Transliterating diverse scripts into a single representation (Latin or ITRANS) bridges the script barrier, allowing one multilingual model to share embeddings across Hindi, Bengali, Marathi, and Gujarati.
3. **Challenges:** Subtle language differences (e.g. Schwa deletion in Hindi vs explicit vowels in Bengali) and conjunct variations require careful phonetic rules.

## Q4. Analyze Parallel Corpora and Handle Code-Mixed Text

In [12]:
# (a) Create 20 parallel sentence pairs (Indic: Hindi and English)
parallel_pairs = [
    ("यह एक सुंदर दिन है।", "It is a beautiful day."),
    ("मुझे नई चीजें सीखना पसंद है।", "I like to learn new things."),
    ("भारत एक विशाल और विविध देश है।", "India is a large and diverse country."),
    ("सूर्य पूर्व से उगता है।", "The sun rises in the east."),
    ("विद्यार्थी पुस्तकालय में पढ़ रहे हैं।", "Students are studying in the library."),
    ("कंप्यूटर हमारे जीवन को आसान बनाते हैं।", "Computers make our lives easier."),
    ("पेड़ हमें शुद्ध हवा देते हैं।", "Trees give us fresh air."),
    ("वह कल दिल्ली जा रहा है।", "He is going to Delhi tomorrow."),
    ("स्वास्थ्य ही सबसे बड़ा धन है।", "Health is the greatest wealth."),
    ("समय बहुत मूल्यवान होता है।", "Time is very valuable."),
    ("बच्चे मैदान में खेल रहे हैं।", "Children are playing in the ground."),
    ("नदी का पानी बहुत साफ है।", "The river water is very clean."),
    ("सफलता कठिन परिश्रम से मिलती है।", "Success comes from hard work."),
    ("हम आज शाम को मिलेंगे।", "We will meet this evening."),
    ("पुस्तकें हमारी सच्ची मित्र होती हैं।", "Books are our true friends."),
    ("बारिश के बाद मौसम सुहावना हो गया।", "The weather became pleasant after the rain."),
    ("विज्ञान ने दुनिया को बदल दिया है।", "Science has changed the world."),
    ("हमेशा सच बोलना चाहिए।", "One should always speak the truth."),
    ("किसान खेतों में मेहनत करते हैं।", "Farmers work hard in the fields."),
    ("संगीत मन को शांति देता है।", "Music gives peace to the mind.")
]

print(f"Total parallel sentence pairs: {len(parallel_pairs)}")


Total parallel sentence pairs: 20


In [13]:
# (b) Tokenize and align the source and target sentences
aligned_data = []
hindi_lengths = []
english_lengths = []
hindi_vocab = set()
english_vocab = set()

for i, (hi, en) in enumerate(parallel_pairs, 1):
    punct_chars = "।,;:!?()[]{}|./«»\'\""
    hi_tokens = [w.strip(punct_chars) for w in hi.split() if w.strip(punct_chars)]
    en_tokens = re.findall(r'[a-zA-Z]+', en.lower())
    
    hindi_lengths.append(len(hi_tokens))
    english_lengths.append(len(en_tokens))
    hindi_vocab.update(hi_tokens)
    english_vocab.update(en_tokens)
    
    aligned_data.append({
        'Pair': f"Pair {i}",
        'Hindi Tokens': hi_tokens,
        'English Tokens': en_tokens,
        'Hindi Len': len(hi_tokens),
        'English Len': len(en_tokens)
    })

aligned_df = pd.DataFrame(aligned_data)
print("First 5 Aligned Sentence Pairs:")
print(aligned_df.head(5)[['Pair', 'Hindi Tokens', 'English Tokens', 'Hindi Len', 'English Len']])


First 5 Aligned Sentence Pairs:
     Pair                                 Hindi Tokens  \
0  Pair 1                     [यह, एक, सुंदर, दिन, है]   
1  Pair 2           [मुझे, नई, चीजें, सीखना, पसंद, है]   
2  Pair 3        [भारत, एक, विशाल, और, विविध, देश, है]   
3  Pair 4                 [सूर्य, पूर्व, से, उगता, है]   
4  Pair 5  [विद्यार्थी, पुस्तकालय, में, पढ़, रहे, हैं]   

                                 English Tokens  Hindi Len  English Len  
0                   [it, is, a, beautiful, day]          5            5  
1             [i, like, to, learn, new, things]          6            6  
2  [india, is, a, large, and, diverse, country]          7            7  
3              [the, sun, rises, in, the, east]          5            6  
4   [students, are, studying, in, the, library]          6            6  


In [14]:
# (c) Calculate statistics: sentence length and vocabulary size for both languages
stats_df = pd.DataFrame({
    'Metric': ['Total Sentences', 'Total Tokens', 'Vocabulary Size', 'Average Sentence Length'],
    'Hindi': [len(parallel_pairs), sum(hindi_lengths), len(hindi_vocab), np.mean(hindi_lengths)],
    'English': [len(parallel_pairs), sum(english_lengths), len(english_vocab), np.mean(english_lengths)]
})

print("Parallel Corpus Statistics:")
print(stats_df)


Parallel Corpus Statistics:
                    Metric  Hindi  English
0          Total Sentences   20.0     20.0
1             Total Tokens  118.0    112.0
2          Vocabulary Size   94.0     84.0
3  Average Sentence Length    5.9      5.6


In [15]:
# (d) Create at least 10 code-mixed sentences (Hinglish)
code_mixed_sentences = [
    "Bro kal exam hai aur maine kuch study nahi kiya.",
    "Yeh movie sach me super amazing thi yaar!",
    "Kya tum college aa rahe ho today?",
    "Traffic bahut heavy tha isliye late ho gaya.",
    "Data Science ka assignment submit karna compulsory hai.",
    "Mujhe lagta hai yeh project easily complete ho jayega.",
    "Aaj weather pleasant hai chalo outing par chalte hain.",
    "Coffee break ke baad meeting start karenge.",
    "New phone ka camera quality really outstanding hai.",
    "Plzz mujhe thoda time do main call back karta hoon."
]

print(f"Total code-mixed sentences created: {len(code_mixed_sentences)}")


Total code-mixed sentences created: 10


In [16]:
# (e) Identify language segments / tokens and apply suitable normalization rules
def process_code_mixed(sentence):
    # Normalize repeated characters (e.g. 'plzz' -> 'plz')
    clean_sent = re.sub(r'(.)\1{2,}', r'\1', sentence)
    tokens = clean_sent.split()
    
    # Common Hindi words in Roman script for simple rule-based LID
    hindi_roman_vocab = {
        'bro', 'kal', 'hai', 'aur', 'maine', 'kuch', 'nahi', 'kiya',
        'yeh', 'sach', 'me', 'thi', 'yaar', 'kya', 'tum', 'aa', 'rahe',
        'ho', 'bahut', 'tha', 'isliye', 'gaya', 'ka', 'karna', 'mujhe',
        'lagta', 'jayega', 'aaj', 'chalo', 'par', 'chalte', 'hain',
        'ke', 'baad', 'karenge', 'do', 'main', 'karta', 'hoon'
    }
    
    tagged_tokens = []
    for token in tokens:
        word = re.sub(r'[^\w]', '', token).lower()
        if not word:
            continue
        if word in hindi_roman_vocab:
            lang = 'HINDI_ROMAN'
        elif re.match(r'^[a-z]+$', word):
            lang = 'ENGLISH'
        else:
            lang = 'OTHER'
        tagged_tokens.append((token, lang))
        
    return clean_sent, tagged_tokens

code_mixed_results = []
for sent in code_mixed_sentences:
    norm_sent, tagged = process_code_mixed(sent)
    code_mixed_results.append({
        'Sentence': norm_sent,
        'Tagged Tokens': tagged
    })

print("Sample Code-Mixed Sentences with Language Identification (LID):")
for item in code_mixed_results[:4]:
    print(f"\nSentence: {item['Sentence']}")
    print(f"Language Tags: {item['Tagged Tokens']}")


Sample Code-Mixed Sentences with Language Identification (LID):

Sentence: Bro kal exam hai aur maine kuch study nahi kiya.
Language Tags: [('Bro', 'HINDI_ROMAN'), ('kal', 'HINDI_ROMAN'), ('exam', 'ENGLISH'), ('hai', 'HINDI_ROMAN'), ('aur', 'HINDI_ROMAN'), ('maine', 'HINDI_ROMAN'), ('kuch', 'HINDI_ROMAN'), ('study', 'ENGLISH'), ('nahi', 'HINDI_ROMAN'), ('kiya.', 'HINDI_ROMAN')]

Sentence: Yeh movie sach me super amazing thi yaar!
Language Tags: [('Yeh', 'HINDI_ROMAN'), ('movie', 'ENGLISH'), ('sach', 'HINDI_ROMAN'), ('me', 'HINDI_ROMAN'), ('super', 'ENGLISH'), ('amazing', 'ENGLISH'), ('thi', 'HINDI_ROMAN'), ('yaar!', 'HINDI_ROMAN')]

Sentence: Kya tum college aa rahe ho today?
Language Tags: [('Kya', 'HINDI_ROMAN'), ('tum', 'HINDI_ROMAN'), ('college', 'ENGLISH'), ('aa', 'HINDI_ROMAN'), ('rahe', 'HINDI_ROMAN'), ('ho', 'HINDI_ROMAN'), ('today?', 'ENGLISH')]

Sentence: Traffic bahut heavy tha isliye late ho gaya.
Language Tags: [('Traffic', 'ENGLISH'), ('bahut', 'HINDI_ROMAN'), ('heavy', '

### Observations on Q4: Challenges of Parallel Data and Code-Mixed Text

1. **Word Order Difference (SOV vs SVO):** Hindi is Subject-Object-Verb, while English is Subject-Verb-Object, complicating token-to-token alignment.
2. **Postpositions vs Prepositions:** Hindi uses postpositions attached after nouns (*घर में*), whereas English uses prepositions before nouns (*in the house*).
3. **No Standard Spelling in Code-Mixing:** Hinglish text exhibits extreme spelling variations (e.g., *kya, kyaa, keya*), requiring phonetic normalization.
4. **Token-Level Ambiguity:** Words like *'to'* (English preposition vs Hindi 'तो') or *'me'* (English pronoun vs Hindi 'में') create language identification collisions.